# 📱 H3 Mobile Studio｜Google Colab Pro 專用手機影音工作站

這是手機版的啟動器：**Colab 做後端**、Gradio 做可安裝至手機主畫面的 **PWA 網頁**。  
**第一次啟動：** ① 選擇 L4 或 A100 GPU → ② 依序執行 3 格程式碼 → ③ 設定至少 10 字元的登入密碼 → ④ 在最後輸出看到 `https://xxxxx.gradio.live` 手機連結。

### 功能
- **MiniMax H3**：從手機上傳動漫角色圖、提示詞、5／10／15 秒影片，包含原生環境音。
- **Real-ESRGAN Anime**：720p／1080p 動漫畫質升級（較慢、逐幀工作）。
- **Qwen3-TTS**：中文語音、角色聲線、情緒描述。
- **字幕、混音、光流補幀**：台詞字幕與 MP4 合成；選擇光流 48fps（**非 RIFE**）。
- **任務佇列 + Google Drive**：提交後可以關掉手機頁面，Colab 還在運作就會繼續執行。完成後保存到 `我的雲端硬碟/H3_Mobile_Studio`，可重新開啟手機頁面取得檔案。

⚠️ **限制**：Colab Pro 不能保證 24 小時常駐或固定 GPU，Google 官方一般執行階段最長約 12 小時；運算單元用完、休眠、網路問題都可能中斷。Gradio 分享連結也會到期或在重啟後改變。這不是永久部署或獨立 APK。  
⚠️ **安全**：最後的公開分享連結須搭配你自行設定的密碼使用；勿將帳號／密碼分享給陌生人。  
⚠️ **商用**：MiniMax H3 開源權重有特定的商用授權要求；公開發布或營利前請核對官方條款。

**來源**：[Colab 官方 FAQ](https://research.google.com/colaboratory/intl/zh-TW/faq.html)｜[Gradio PWA 與 share](https://gradio.app/guides/sharing-your-app)｜[H3 官方教學](https://docs.comfy.org/tutorials/video/minimax/minimax-h3)


In [ ]:
# ① 安裝手機版介面與基本依賴
import sys, subprocess, torch
subprocess.run(
    [sys.executable,"-m","pip","install","-q","gradio>=5.40,<6",
     "huggingface_hub[hf_xet]","requests","Pillow","soundfile"],
    check=True)
print("✅ 介面套件已安裝")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "未取得 GPU，請先切換執行階段")


In [ ]:
# ② 掛載 Google Drive，下載已製作好的手機版程式（含語法檢查）
from google.colab import drive
from pathlib import Path
import requests, importlib.util

drive.mount("/content/drive")
url = "https://raw.githubusercontent.com/eric931130/aiagent/main/colab/h3_mobile_studio.py"
response=requests.get(url,timeout=90)
response.raise_for_status()

path=Path("/content/h3_mobile_studio.py")
path.write_text(response.text,encoding="utf-8")
compile(response.text,str(path),"exec")
spec=importlib.util.spec_from_file_location("h3_mobile_studio",path)
h3mobile=importlib.util.module_from_spec(spec)
spec.loader.exec_module(h3mobile)
print("✅ 手機工作站程式已載入：",path)
print("✅ 顯卡：",h3mobile.gpu_state())
print("✅ 模型：",h3mobile.model_state())


In [ ]:
# ③ 啟動手機版網站｜請自己設定密碼
from getpass import getpass

username="studio"
password=getpass("請設定手機登入密碼（至少 10 個字元，不會顯示在畫面上）：")
if len(password)<10:
    raise ValueError("請使用至少 10 字元的密碼，再重跑此儲存格")

# 執行後畫面會產生 https://xxxxx.gradio.live
# 可用手機 Chrome 開啟，輸入 studio 與你剛設定的密碼。
# 當 Colab GPU 執行階段終止，網址就會失效，需要重新執行此格。
h3mobile.launch(username,password)
